# Introduction
This notebook will use PySpark's MLlib to build and compare supervised learning models on a gene
expression dataset for leukemia classification. First, we read in the data, split it into training and test
sets, and define an evaluation metric. We then fit three model classes (Random Forest, Gradient-Boosted
Tree, and Multilayer Perceptron) using pipelines and cross-validation to select the best
hyperparameters for each. Finally, we evaluate each best model on the held-out test set and determine
an overall winner.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = (SparkSession.builder
    .appName("GeneExpression")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 14:53:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Dataset Overview

**Source:** [Gene Expression Dataset](https://www.kaggle.com/datasets/ziya07/gene-expression-dataset) by Ziya (Kaggle)

This dataset contains microarray gene expression profiles curated for building
computational models for early leukemia diagnosis. Each row is a patient sample
whose columns are normalised expression levels of individual genes obtained from
microarray experiments. The target column (last column) assigns each sample to one
of three diagnostic classes:

- **ALL** — Acute Lymphoblastic Leukemia
- **AML** — Acute Myeloid Leukemia
- **Healthy** — Healthy Controls

According to the dataset card the data has already been preprocessed:
* Missing values have been imputed.
* Normalisation has been applied so gene expression values are uniformly scaled.

Below is quick overview of the dimensions, schema, class balance, and presence of any
remaining null values.

In [3]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("leukemia_gene_expression.csv")

# Identify columns
label_col = df.columns[-1]
feature_cols = [c for c in df.columns if c != label_col]

In [10]:
from pyspark.sql.functions import col, sum as _sum, when
from pyspark.sql.types import DoubleType

# Dimensions
print(f"Rows:    {df.count()}")
print(f"Columns:  {len(df.columns)}")
print(f"Features: {len(feature_cols)}")
print(f"Label:    '{label_col}'\n")

# Check for non-double feature columns
non_double = [(f.name, f.dataType) for f in df.schema.fields
              if f.name in feature_cols and not isinstance(f.dataType, DoubleType)]

if non_double:
    print(f"Feature columns that are NOT DoubleType ({len(non_double)}):")
    for name, dtype in non_double:
        print(f"  {name: } {dtype}")
    print()
else:
    print("All feature columns are DoubleType.\n")

# Class distribution
print(f"Class distribution '{label_col}':")
df.groupBy(label_col).count().orderBy("count", ascending=False).show()

# Missing / null values
null_row = df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).first()

bad = [(c, null_row[c]) for c in df.columns if null_row[c] > 0]

if bad:
    print(f"Columns with nulls ({len(bad)}):")
    for name, cnt in bad[:10]:
        print(f"  {name: } {cnt}")
else:
    print("No missing or NaN values found in any column.\n")

Rows:    1000
Columns:  1001
Features: 1000
Label:    'Diagnosis'

All feature columns are DoubleType.

Class distribution 'Diagnosis':
+---------+-----+
|Diagnosis|count|
+---------+-----+
|      AML|  409|
|      ALL|  380|
|  Healthy|  211|
+---------+-----+

No missing or NaN values found in any column.



In [5]:
# Train/Test split
train, test = df.randomSplit([0.8, 0.2], seed=42)

# Pipeline: 4 transformations + 1 estimator
label_indexer = StringIndexer(inputCol=label_col, outputCol="label")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withStd=True, withMean=True)
pca = PCA(inputCol="features_scaled", outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

pipeline = Pipeline(stages=[label_indexer, assembler, scaler, pca, rf])

# Cross-validation
param_grid = (ParamGridBuilder()
    .addGrid(pca.k, [10, 30, 50])
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=param_grid,
                    evaluator=evaluator, numFolds=5, seed=42, parallelism=16)

cv_model = cv.fit(train)

# Evaluate
rf_predictions = cv_model.transform(test)
print(f"Random Forest — Test F1: {evaluator.evaluate(rf_predictions):.4f}")

# Best hyperparameters
best = cv_model.bestModel
print(f"Best PCA k:    {best.stages[3].getK()}")
print(f"Best numTrees: {best.stages[4].getNumTrees}")
print(f"Best maxDepth: {best.stages[4].getOrDefault('maxDepth')}")

Random Forest — Test F1: 0.3833
Best PCA k:    10
Best numTrees: 200
Best maxDepth: 15
